# Thực hành BB84: từ phép đo đến phân tích kết quả

Chạy các ô theo thứ tự từ trên xuống. Notebook dùng NumPy/Matplotlib, không yêu cầu Qiskit, máy tính lượng tử hoặc dịch vụ đám mây. Thời gian chạy các phép quét nhỏ thường ngắn hơn bộ thí nghiệm CLI.

**Mục tiêu:** giải thích sifting, QBER 25% của intercept–resend, tác động của nhiễu và bất định lấy mẫu. Đây là mô phỏng giảng dạy, chưa triển khai xác thực, sửa lỗi, kiểm tra khóa hay privacy amplification; không tạo khóa bí mật sử dụng được.

Cài môi trường theo README rồi mở notebook bằng Python của `.venv`. Quy ước trong source: cơ sở 0 = Z, 1 = X. Xem `docs/SCIENCE.md` cho các giả định và tài liệu tham khảo.

In [ ]:
from pathlib import Path
import sys

# Hoạt động khi notebook được mở từ thư mục dự án hoặc thư mục notebooks/.
working = Path.cwd().resolve()
root = next((p for p in [working, *working.parents]
             if (p / 'bb84' / '__init__.py').is_file()
             and (p / 'requirements.txt').is_file()), None)
if root is None:
    raise RuntimeError('Hãy mở Jupyter từ thư mục dự án hoặc thư mục notebooks của dự án.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
from IPython import get_ipython
# Hiển thị hình trong notebook, kể cả khi môi trường đặt backend Agg.
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
from bb84 import BB84Config, simulate

print('Thư mục dự án:', root)
print('Python:', sys.executable)
print('NumPy:', np.__version__)
SEED = 84  # Seed chỉ để tái lập mô phỏng, không dùng tạo khóa mật mã.


## 1. Ba kịch bản đầu tiên

Chọn cơ sở Z/X với xác suất bằng nhau. Khi không mất tín hiệu, số bit qua sifting gần một nửa số tín hiệu phát. Mẫu công bố để kiểm tra lỗi bị loại khỏi phần dữ liệu còn lại.

- **Lý tưởng:** không có Eve hoặc nhiễu, QBER bằng 0.
- **Nhiễu:** với kênh depolarizing $\rho\mapsto(1-p)\rho+pI/2$, $p=0.08$ cho QBER kỳ vọng 0.04.
- **Eve chặn toàn bộ:** Eve chọn cơ sở đều; xác suất sai trên chuỗi sifted là $(1/2)(1/2)=1/4$.

`qber_actual` là tỷ lệ lỗi toàn bộ sifted data **do simulator biết**; `qber_estimate` chỉ dùng mẫu công bố. QBER thực nghiệm dao động; không buộc phải đúng giá trị kỳ vọng trong từng lần chạy. `continue_postprocessing` không có nghĩa đã có khóa an toàn.

In [ ]:
cases = {
    'Lý tưởng': BB84Config(n_signals=10_000, sample_fraction=0.2),
    'Nhiễu p=0.08': BB84Config(n_signals=10_000, noise_model='depolarizing',
                             noise_probability=0.08, sample_fraction=0.2),
    'Eve f=1': BB84Config(n_signals=10_000, eve_fraction=1.0, sample_fraction=0.2),
}
summaries = {name: simulate(cfg, seed=SEED + i).summary()
             for i, (name, cfg) in enumerate(cases.items())}
for name, s in summaries.items():
    print('\n' + name)
    print('  Phát / detected / sifted / mẫu / còn lại:',
          s['n_signals'], s['n_detected'], s['n_sifted'], s['n_test'], s['n_remaining'])
    print('  QBER simulator / mẫu:', s['qber_actual'], s['qber_estimate'])
    print('  Khoảng Wilson:', s['qber_interval'])
    print('  Quyết định giảng dạy:', s['decision'])

# Kiểm tra tính chất xác định của kênh lý tưởng, không kiểm tra một tỷ lệ ngẫu nhiên bằng đúng 1/2.
assert summaries['Lý tưởng']['actual_errors'] == 0
assert all(s['n_test'] + s['n_remaining'] == s['n_sifted'] for s in summaries.values())


## 2. Nhìn từng tín hiệu trong ví dụ 16 bit

Bảng sau cố ý công khai toàn bộ dữ liệu để học thao tác; trong một giao thức thực không được công khai toàn bộ bit Alice/Bob như vậy. `sift` chỉ đúng khi tín hiệu được phát hiện và cơ sở hai bên trùng; `mẫu` đánh dấu bit công bố; `còn` đánh dấu phần sau khi bỏ mẫu.

Chuỗi rất ngắn có thể thiếu mẫu hoặc cho 0 lỗi do ngẫu nhiên. Không suy ra mức an toàn từ ví dụ này. Thử đổi `seed` và giải thích các dòng thay đổi.

In [ ]:
toy = simulate(BB84Config(n_signals=16, eve_fraction=0.5,
                          loss_probability=0.1, sample_fraction=0.25), seed=SEED)
basis = {0: 'Z', 1: 'X'}
print(' i  A  cơ_sở_A  cơ_sở_B  B   Eve   nhận  sift  mẫu  còn')
for i in range(16):
    bob = str(int(toy.bob_bits[i])) if toy.detected_mask[i] else '—'
    eve = basis[int(toy.eve_bases[i])] if toy.eve_mask[i] else '—'
    print(f'{i:2d}  {int(toy.alice_bits[i])}      {basis[int(toy.alice_bases[i])]}        '
          f'{basis[int(toy.bob_bases[i])]}     {bob:>1}    {eve:>1}      '
          f'{int(toy.detected_mask[i])}     {int(toy.sift_mask[i])}     '
          f'{int(toy.test_mask[i])}    {int(toy.key_mask[i])}')
print('\nTóm tắt:', toy.summary())
assert not np.any(toy.test_mask & toy.key_mask)


## 3. Tự kiểm chứng quan hệ $Q=f/4$

Mỗi cấu hình chạy 10 lần độc lập, 4.000 tín hiệu/lần. Hình dùng QBER toàn chuỗi sifted làm chẩn đoán mô phỏng, không phải ước lượng giao thức. Thanh sai số là **một độ lệch chuẩn giữa các lần chạy**, không phải khoảng tin cậy 95% hay security bound.

Câu hỏi: khi tăng số tín hiệu, độ dao động giữa các lần chạy thay đổi thế nào? Nếu tăng số lần lặp, độ chính xác của trung bình thay đổi ra sao? Hai thao tác này không đồng nghĩa tăng số bit trong từng lần thực hiện giao thức.

In [ ]:
fractions = np.linspace(0.0, 1.0, 11)
repeats = 10
means, spreads = [], []
for j, f in enumerate(fractions):
    values = []
    for rep in range(repeats):
        cfg = BB84Config(n_signals=4_000, eve_fraction=float(f), sample_fraction=0.2)
        s = simulate(cfg, seed=SEED + 1000 * j + rep).summary()
        values.append(s['qber_actual'])
    means.append(np.mean(values))
    spreads.append(np.std(values, ddof=1))

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.errorbar(fractions, means, yerr=spreads, fmt='o', capsize=3,
            label='Monte Carlo: trung bình ± 1 SD')
ax.plot(fractions, fractions / 4, label='Kỳ vọng: Q = f/4')
ax.set(xlabel='Tỷ lệ Eve chặn f', ylabel='QBER toàn chuỗi sifted (simulator)',
       title='Intercept–resend, không nhiễu, không mất tín hiệu')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()


## 4. Cỡ mẫu và điều chúng ta chưa biết

Giữ $f=0.2$ nên QBER kỳ vọng là 0.05; so sánh các cỡ mẫu 20, 100 và 500. Mỗi dòng là một lần thực hiện có seed riêng. Khoảng Wilson minh họa bất định tỷ lệ lỗi; nó có thể không bao phủ giá trị kỳ vọng trong một số lần chạy và không phải cận an toàn finite-key composable.

Mẫu lớn hơn thường làm khoảng hẹp hơn nhưng tiêu tốn nhiều bit sifted. Xác suất có ít nhất một lỗi trong mẫu i.i.d. là $1-(1-Q)^m$; đây **khác** xác suất chương trình trả về `abort`, và thấy lỗi không tự xác định được Eve.

In [ ]:
for j, m in enumerate([20, 100, 500]):
    cfg = BB84Config(n_signals=5_000, eve_fraction=0.2, sample_size=m,
                     confidence=0.95, abort_threshold=0.11)
    s = simulate(cfg, seed=SEED + 20_000 + j).summary()
    print(f"m={s['n_test']:3d}, lỗi mẫu={s['test_errors']:3d}, "
          f"Q mẫu={s['qber_estimate']:.4f}, Wilson={s['qber_interval']}, "
          f"còn lại={s['n_remaining']}, trạng thái={s['decision']}")
    print(f'  P(thấy ít nhất 1 lỗi), mô hình i.i.d. Q=0.05: {1 - 0.95**m:.6f}')


## 5. Đường tham chiếu entropy và mốc 11%

Với nguồn qubit lý tưởng, khóa dài vô hạn, lỗi hai cơ sở đối xứng, hậu xử lý một chiều không noisy preprocessing và sửa lỗi lý tưởng, một bound đạt được là $r_\infty=[1-2h_2(Q)]_+$. Với chi phí sửa lỗi tham chiếu $f_{EC}>1$, thay bằng $[1-h_2(Q)-f_{EC}h_2(Q)]_+$. $r$ ở đây tính trên mỗi bit khóa thô dùng tạo khóa, không phải mỗi tín hiệu phát.

Hình dưới là **tính công thức lý thuyết**, không phải khóa bí mật được mô phỏng trích xuất. QBER dưới 11% hoặc quyết định `continue_postprocessing` không tự chứng minh an toàn. Cần xác thực, ước lượng thống kê phù hợp, EC/verification, privacy amplification và một theorem đúng với thiết bị/giao thức.

Nguồn: [Shor–Preskill (2000)](https://doi.org/10.1103/PhysRevLett.85.441), [Scarani et al. (2009)](https://doi.org/10.1103/RevModPhys.81.1301), [Cai–Scarani (2009)](https://doi.org/10.1088/1367-2630/11/4/045024). Để chạy bộ sáu thí nghiệm và xuất CSV/PNG/PDF/HTML, dùng lệnh `experiments` trong README.

In [ ]:
from bb84.theory import binary_entropy, asymptotic_secret_fraction

q = np.linspace(0.0, 0.2, 401)
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for f_ec in [1.0, 1.1, 1.2]:
    reference = asymptotic_secret_fraction(q, f_ec=f_ec)
    ax.plot(q, reference, label=f'f_EC = {f_ec:.1f}')
ax.axvline(0.1100, color='gray', ls='--', alpha=0.7, label='≈11%, f_EC=1')
ax.set(xlabel='QBER Q (hai cơ sở đối xứng)',
       ylabel='Tỷ lệ tiệm cận tham chiếu / bit khóa thô',
       title='Bound tham chiếu; chưa thực hiện trích xuất khóa')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()
assert np.allclose(binary_entropy(np.array([0.0, 0.5, 1.0])), [0.0, 1.0, 0.0])


## 6. Phân biệt nhiễu và mất tín hiệu

Cùng tham số $p=0.08$, kênh depolarizing của dự án gây QBER kỳ vọng $p/2=0.04$, còn lỗi đảo đầu ra Bob gây QBER $p=0.08$. Đây là hai kênh khác nhau. Mất tín hiệu độc lập không tự đảo bit; nó giảm số vị trí được giữ lại.

Hãy kiểm tra mẫu số của từng tỷ lệ. Khi $p_Z=1/2$ và tỷ lệ mất $L=0.3$, tỷ lệ sifted trên số tín hiệu phát có kỳ vọng $(1-L)/2=0.35$. Dữ liệu gộp dưới đây dùng chẩn đoán của simulator.


In [ ]:
from bb84.theory import expected_qber, expected_sift_fraction

for i, model in enumerate(['depolarizing', 'readout_flip']):
    cfg = BB84Config(n_signals=20_000, noise_probability=0.08,
                     noise_model=model, loss_probability=0.3)
    s = simulate(cfg, seed=SEED + 30_000 + i).summary()
    print(model)
    print('  QBER mô phỏng / kỳ vọng:', s['qber_actual'],
          expected_qber(noise_probability=0.08, noise_model=model))
    print('  Sifted/phát mô phỏng / kỳ vọng:', s['sift_fraction'],
          expected_sift_fraction(p_z=0.5, loss_probability=0.3))

# Bài tập: thêm eve_fraction=0.5 rồi so sánh với expected_qber(0.5, 0.08, model).
# Vì sao không được cộng đơn giản hai xác suất lỗi?


## 7. Kiểm chứng bằng mạch Qiskit — tùy chọn

Phần chính của notebook đã hoàn thành mà không cần Qiskit. Nếu đã cài `requirements-optional.txt`, đổi `RUN_QISKIT` thành `True` để đo tám tổ hợp (hai cơ sở chuẩn bị × hai bit × hai cơ sở đo) trên AerSimulator cục bộ. Không cần token IBM hoặc kết nối đến phần cứng lượng tử.

Nguồn API: [AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html). Tài liệu thực hành tham khảo: [IBM Quantum — Quantum key distribution](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/quantum-key-distribution). Các mạch một qubit này kiểm chứng xác suất đo, không phải một đường truyền QKD vật lý giữa Alice và Bob.


In [ ]:
RUN_QISKIT = False
if RUN_QISKIT:
    from bb84.qiskit_demo import run_qiskit_check
    records = run_qiskit_check(shots=4096, seed=SEED)
    for row in records:
        print(row['preparation_basis'], row['alice_bit'], '→',
              row['measurement_basis'], row['counts'],
              'P(1) kỳ vọng:', row['expected_p_one'])
    print('\nVí dụ mạch:\n', records[-1]['circuit'])
else:
    print('Bỏ qua Qiskit tùy chọn. Đổi RUN_QISKIT=True khi đã cài phụ thuộc.')


## 8. Từ notebook đến kết quả nộp bài

Notebook dùng phép quét nhỏ để hiểu phương pháp. Để có sáu hình PNG/PDF, dữ liệu CSV, metadata và báo cáo HTML, chạy lệnh `experiments --preset standard` trong README.

Trước khi kết luận, tự trả lời: (1) QBER này tính từ toàn chuỗi hay mẫu công bố? (2) thanh sai số là SD hay khoảng tin cậy? (3) mẫu số tỷ lệ là tín hiệu phát hay bit sifted? (4) tấn công, kênh và giả định thiết bị nào được xét? (5) chương trình đã thực hiện xác thực, EC và PA chưa?

Nộp kèm mã, cấu hình/seed, metadata, CSV và phân tích ít nhất ba đồ thị. Không dùng chuỗi bit từ mô phỏng này làm khóa mật mã.
